# Normal Execution

In [ ]:
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, Literal
from langgraph.graph import START, END, StateGraph
from openai import OpenAI
import json
from groq import Groq
from pydantic import BaseModel, Field

In [ ]:
load_dotenv(find_dotenv())

In [ ]:
class InputAnalyser(TypedDict):
    user_input: str
    analyse_output: str
    platforms: Literal['youtube', 'social_media', 'blog_posts', 'news_articles']

In [ ]:
class PlatformRequired(BaseModel):
    analysis: str
    platform_needed: list[Literal['youtube', 'social_media', 'blog_posts', 'news_articles']] = Field(..., description="The values with which the the information can be extracted")

In [ ]:
groq_client = Groq()

In [ ]:
for model in groq_client.models.list().data:
    print(f"Model name: {model.id}")

In [ ]:
SYSTEM_PROMPT = """
Role: You are a powerful and intelligent examiner. 

Task: You will be given an input in the form of string and you will then analyse the input and provide your analysis about what the user is asking. 
After that you have to tell, from which platform the information can be extracted which can information about the user query.

There can be multiple platform from which the information can be extracted e.g blog sites, youtube videos.

You have below list of platform:
1. Youtube
2. Blog Post platform like Medium, Dev.to
3. Social media platform like Linkedin, X.com
4. News Article platform like Dainik Jagran (jagran.com), New York Times (nytimes.com)

Output: I need the final output which have 2 values:
1. Analysis: Your analysis about the user input query and why you have choose that or these platforms.
2. Platform Required: which can be single platform or a list of platform using which, information can be provided to the user
"""

In [ ]:
def model_completion(system_prompt: str, user_prompt: str) -> str | None:

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        model="meta-llama/llama-4-maverick-17b-128e-instruct",
        temperature=0,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "PlatformRequired",
                "schema": PlatformRequired.model_json_schema()
            }
        }
    )

    return chat_completion.choices[0].message.content

In [ ]:
def analyse_input(state: InputAnalyser):
    user_input = state['user_input']

    model_response = json.loads(model_completion(system_prompt=SYSTEM_PROMPT, user_prompt=user_input)) # type: ignore

    if user_input:
        return {'analyse_output': model_response['analysis'], 'platforms': model_response['platform_needed']}
    else:
        return {'analyse_output': "", 'platforms': None}

In [ ]:
builder = StateGraph(InputAnalyser)
builder.add_node("input_analyser", analyse_input)
builder.add_edge(START, "input_analyser")
builder.add_edge("input_analyser", END)

graph = builder.compile()

graph

In [ ]:
graph.invoke({"user_input": "what is the latest update happening between India and Pakistan related to cricket"}) # type: ignore

# Async Execution with Tools

#### With Tools

In [1]:
# Replace the failing notebook cell with this

import sys
from pathlib import Path

# Find project root (directory that contains "src")
cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src").exists()), None)

if project_root is None:
    raise ModuleNotFoundError("Could not locate project root containing a 'src' directory.")

# Ensure Python can import from project root
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from src.scrappers.blog_posts_scrapper import BlogPostsScraper
from src.scrappers.youtube_scrapper import YouTubeScrapper
from src.scrappers.social_media_scrapper import SocialMediaScraper
from groq import AsyncGroq
from src.config import get_settings
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, Literal
from langgraph.graph import START, END, StateGraph
from pydantic import BaseModel, Field
from IPython.display import display, Markdown

In [3]:
load_dotenv(find_dotenv(), override=True)

True

In [4]:
blog_posts_scraper = BlogPostsScraper()
youtube_scrapper = YouTubeScrapper()
social_media_scrapper = SocialMediaScraper()

2026-03-08 13:37:50,967 | blog_posts_scrapper.py:67 | INFO | Initialized BlogPostsScraper with Tavily client.
2026-03-08 13:37:50,968 | social_media_scrapper.py:52 | INFO | Initialized SocialMediaScraper (max_results=5, days=3).


In [ ]:
class InputAnalyser(TypedDict):
    user_input: str
    analyse_output: str
    platforms: Literal['youtube', 'social_media', 'blog_posts', 'news_articles']

In [ ]:
class PlatformRequired(BaseModel):
    analysis: str
    platform_needed: list[Literal['youtube', 'social_media', 'blog_posts', 'news_articles']] = Field(..., description="The values with which the the information can be extracted")

In [ ]:
SYSTEM_PROMPT = """
Role: You are a powerful and intelligent examiner. 

Task: You will be given an input in the form of string and you will then analyse the input and provide your analysis about what the user is asking. 
After that you have to tell, from which platform the information can be extracted which can information about the user query.

There can be multiple platform from which the information can be extracted e.g blog sites, youtube videos.

You have below list of platform:
1. Youtube
2. Blog Post platform like Medium, Dev.to
3. Social media platform like Linkedin, X.com
4. News Article platform like Dainik Jagran (jagran.com), New York Times (nytimes.com)

Output: I need the final output which have 2 values:
1. Analysis: Your analysis about the user input query and why you have choose that or these platforms.
2. Platform Required: which can be single platform or a list of platform using which, information can be provided to the user
"""

In [ ]:
async def get_groq_client() -> AsyncGroq:
    """Get or create groq client in singleton manner
    This is created once for all the subsequent processes
    Returns:
        AsyncGroq: async groq client object
    """
    return AsyncGroq(api_key=get_settings().GROQ_API_KEY.get_secret_value())

In [ ]:
async def async_model_completion(system_prompt: str, user_prompt: str) -> str | None:
    groq_client = await get_groq_client()

    chat_completion = await groq_client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        model="meta-llama/llama-4-maverick-17b-128e-instruct",
        temperature=0,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "PlatformRequired",
                "schema": PlatformRequired.model_json_schema()
            }
        }
    )

    return chat_completion.choices[0].message.content

In [ ]:
async def analyse_input(state: InputAnalyser) -> dict:
    user_input = state["user_input"]
    if not user_input:
        return {"analyse_output": "", "platforms": None}

    model_response = await async_model_completion(
        system_prompt=SYSTEM_PROMPT, user_prompt=user_input
    )

    if not model_response:
        return {"analyse_output": "", "platforms": None}

    json_resp = json.loads(model_response)  # type: ignore

    if json_resp:
        return {
            "analyse_output": json_resp["analysis"],
            "platforms": json_resp["platform_needed"],
        }
    else:
        return {}

In [ ]:
async def build_graph():
    builder = StateGraph(InputAnalyser)
    builder.add_node("input_analyser", analyse_input)
    builder.add_edge(START, "input_analyser")
    builder.add_edge("input_analyser", END)

    graph = builder.compile()

    return graph

In [ ]:
graph = await build_graph()
result = await graph.ainvoke(
        {
            "user_input": "Latest AI trends in 2024"
        }
    )

print(f"Result: {result}")

In [ ]:
await youtube_scrapper.get_transcripts_for_query(query="Latest AI trends in 2024")

In [ ]:
await blog_posts_scraper.search(topic="Latest AI trends in 2024")

In [5]:
social_scrapped_results = await social_media_scrapper.fetch_social_pulse(topic="Latest AI trends in 2024")

2026-03-08 13:37:58,031 | social_media_scrapper.py:66 | INFO | Searching Reddit via Tavily for: site:reddit.com Latest AI trends in 2024
2026-03-08 13:37:58,034 | social_media_scrapper.py:118 | INFO | Searching LinkedIn via Tavily for: site:linkedin.com/posts Latest AI trends in 2024
2026-03-08 13:38:05,350 | social_media_scrapper.py:156 | INFO | LinkedIn search returned 5 items.
2026-03-08 13:38:10,781 | social_media_scrapper.py:104 | INFO | Reddit search returned 5 items.
2026-03-08 13:38:10,785 | social_media_scrapper.py:177 | INFO | Collected 10 total social items.


In [ ]:
for result in social_scrapped_results:
    if result.source.lower() == "linkedin":
        print(result.url)
        display(Markdown(f"**LinkedIn Result:** {result.content}"))
        print("\n---\n")

In [ ]:
for result in social_scrapped_results:
    if result.source.lower() == "reddit":
        print(result.url)
        display(Markdown(f"**Reddit Result:** {result.content}"))